# 02 — Results Analysis

Cross-algorithm comparison of fine-tuning results.

Covers:
- Schema compliance comparison across algorithms
- Chart-type accuracy (top-1, top-3, macro-F1)
- Latency comparison (avg, p50, p95)
- Training loss curves
- Base model vs. fine-tuned model delta
- Robustness metrics (if available)

In [ ]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUTS_DIR = PROJECT_ROOT / 'outputs' / 'experiments'
print(f'Outputs directory: {OUTPUTS_DIR}')
print(f'Experiments found: {len(list(OUTPUTS_DIR.glob("*"))) if OUTPUTS_DIR.exists() else 0}')

In [ ]:
# Load all metrics.json files
def load_all_metrics(outputs_dir: Path) -> dict:
    """Return dict of {experiment_id: metrics_dict} for all completed experiments."""
    results = {}
    if not outputs_dir.exists():
        return results
    for exp_dir in sorted(outputs_dir.iterdir()):
        metrics_file = exp_dir / 'metrics.json'
        if metrics_file.exists():
            with open(metrics_file) as f:
                results[exp_dir.name] = json.load(f)
    return results

all_metrics = load_all_metrics(OUTPUTS_DIR)
print(f'Loaded metrics from {len(all_metrics)} experiments')
for exp_id in all_metrics:
    algo = all_metrics[exp_id].get('algorithm', exp_id.split('_')[0])
    print(f'  {exp_id}  [{algo}]')

In [ ]:
# Schema compliance comparison table
print(f'{"Experiment":<45} {"json_parse%":>11} {"schema%":>9} {"completeness":>13}')
print('-' * 82)

for exp_id, metrics in all_metrics.items():
    ft = metrics.get('eval', {}).get('finetuned_model', {})
    schema = ft.get('schema', {})
    print(
        f'{exp_id:<45} '
        f'{schema.get("json_parse_rate", "N/A"):>11} '
        f'{schema.get("schema_validity_rate", "N/A"):>9} '
        f'{schema.get("completeness_score", "N/A"):>13}'
    )

In [ ]:
# Latency comparison
print(f'{"Experiment":<45} {"avg_s":>8} {"p50_s":>8} {"p95_s":>8}')
print('-' * 72)

for exp_id, metrics in all_metrics.items():
    ft = metrics.get('eval', {}).get('finetuned_model', {})
    lat = ft.get('latency', {})
    print(
        f'{exp_id:<45} '
        f'{lat.get("avg_latency_s", "N/A"):>8} '
        f'{lat.get("p50_latency_s", "N/A"):>8} '
        f'{lat.get("p95_latency_s", "N/A"):>8}'
    )

In [ ]:
# Training loss comparison
print('Training loss by experiment:')
for exp_id, metrics in all_metrics.items():
    loss = metrics.get('train_loss', 'N/A')
    epochs = metrics.get('epoch', 'N/A')
    print(f'  {exp_id:<45}  loss={loss}  epochs={epochs}')

In [ ]:
# Base vs. fine-tuned delta
print('Schema validity: base vs. fine-tuned delta:')
print(f'{"Experiment":<45} {"base%":>7} {"finetuned%":>11} {"delta%":>8}')
print('-' * 75)

for exp_id, metrics in all_metrics.items():
    eval_section = metrics.get('eval', {})
    base_schema  = eval_section.get('base_model',      {}).get('schema', {})
    ft_schema    = eval_section.get('finetuned_model', {}).get('schema', {})
    base_val = base_schema.get('schema_validity_rate', None)
    ft_val   = ft_schema.get('schema_validity_rate', None)
    delta    = round(ft_val - base_val, 2) if (base_val is not None and ft_val is not None) else 'N/A'
    print(f'{exp_id:<45} {str(base_val):>7} {str(ft_val):>11} {str(delta):>8}')